<a href="https://colab.research.google.com/github/esero88/eser-portfolio/blob/main/pension-retention-analytics/notebooks/01_egm_veri_toplama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EGM Veri Toplama — BES Kalıcılık ve Rekabet Projesi

Emeklilik Gözetim Merkezi'nin (egm.org.tr) yayınladığı BES istatistik raporlarını
tarih tarih indirir. Her rapor tek bir tarihin anlık görüntüsüdür; zaman serisi
için tarih listesindeki her tarih tek tek geziliyor.

**Hedef raporlar**

| Kod | Rapor | Ne için |
|---|---|---|
| `GR-01-015` | Sözleşmelerin Yürürlüğe Giriş Ayına Göre Sonlanma Oranı | Kohort kalıcılık eğrisi (2003.10 →) |
| `GR-01-010` | Sonlanan Sözleşmelerin Sonlanma Nedenine Göre Dağılımı | Ayrılma nedenleri kırılımı |
| `GR-01-019` | Cayma Hakkı Kullanım Oranı | Cayma trendi |
| `EGM-05-006` | Emeklilik Şirketi ve Faiz Detayında Fon Büyüklükleri | Şirket bazında pazar payı |
| `EGM-05-001` | Fon Bazında Fon İşletim Gideri Kesinti Oranları | Maliyet rekabeti |

**Akış:** Kurulum → Script → Keşif (`--probe`) → Küçük deneme → Tam toplama → İndir

---

> **Kaynak:** Bu defterde toplanan tüm istatistiksel veriler Emeklilik Gözetim
> Merkezi A.Ş.'ye (EGM) aittir. Kaynak gösterilmeden alıntı yapılamaz.

## 1. Kurulum

Playwright ve Chromium kuruluyor. EGM sayfası JavaScript ile render edildiği için
`requests` boş tablo döndürür; gerçek bir tarayıcı motoru gerekiyor.

Colab oturumu sıfırlandığında bu hücre tekrar çalıştırılmalı.

In [ ]:
!pip install playwright -q
!playwright install chromium
!playwright install-deps chromium

Installing dependencies...
Hit:1 http://security.ubuntu.com/ubuntu noble-security InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Fetched 3,917 B in 1s (4,694 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libasound2t64 is already the newest version (1.2.11-1ubuntu0.3).
libatk-bridge2.0-0t64 is already t

## 2. Toplayıcı script'i

Script repo'da (`scripts/egm_collector.py`) duruyor; buraya kopyalanmak yerine
oradan çekiliyor. Tek doğru nüsha repo olduğu için defterdeki kopya eskimiyor.

Ayrı bir dosya olarak çalıştırılmasının nedeni: Playwright'ın sync API'si
notebook'un asyncio döngüsüyle çakışıyor. `!python` ile ayrı süreç olarak
çalıştırıldığında sorun kalmıyor.

In [ ]:
!wget -q -O egm_collector.py https://raw.githubusercontent.com/esero88/eser-portfolio/main/pension-retention-analytics/scripts/egm_collector.py
!wc -l egm_collector.py

368 egm_collector.py


## 3. Keşif

`--probe` hiçbir şey indirmez; sayfayı açıp iki şeyi raporlar:

- Tarih listesi bulundu mu, kaç tarih var
- "Excel'e Aktar" düğmesi görülebiliyor mu

`RAPOR_URL` değişkenine raporun EGM'deki adresi yazılıyor.

In [ ]:
RAPOR_URL = "https://egm.org.tr/bilgi-merkezi/istatistikler?r=b5eea773-1617-4344-8e66-1771226ee845"   # <-- raporun linkini buraya yapıştır
RAPOR_KODU = "GR-01-015"

assert RAPOR_URL, "Önce RAPOR_URL'i doldur"
!python egm_collector.py --url "$RAPOR_URL" --code "$RAPOR_KODU" --probe

Aciliyor: https://egm.org.tr/bilgi-merkezi/istatistikler?r=b5eea773-1617-4344-8e66-1771226ee845
83 tarih bulundu.  Ilk: '29 Eylül 2026 Salı'  Son: '30 Aralık 2018 Pazar'
Excel dugmesi: BULUNDU

--- ilk 10 tarih ---
   29 Eylül 2026 Salı -> 2026-09-29
   30 Ağustos 2026 Pazar -> 2026-08-30
   30 Temmuz 2026 Perşembe -> 2026-07-30
   29 Haziran 2026 Pazartesi -> 2026-06-29
   30 Mayıs 2026 Cumartesi -> 2026-05-30
   29 Nisan 2026 Çarşamba -> 2026-04-29
   30 Mart 2026 Pazartesi -> 2026-03-30
   27 Şubat 2026 Cuma -> 2026-02-27
   30 Ocak 2026 Cuma -> 2026-01-30
   30 Aralık 2025 Salı -> 2025-12-30

Kesif modu, indirme yapilmadi.


### Çıktının okunması

| Çıktı | Anlamı |
|---|---|
| "N tarih bulundu" + "Excel düğmesi: BULUNDU" | Yolunda, toplamaya geçilebilir |
| "tarih listesi bulunamadı" | Tarih seçici normal bir `<select>` değil; `tarih_secicisini_bul()` güncellenmeli |
| "Excel düğmesi: BULUNAMADI" | `export_dugmesini_bul()` güncellenmeli |
| Zaman aşımı / bağlantı hatası | EGM, Colab'ın IP'sini engelliyor olabilir; script yerel makinede çalıştırılmalı |

## 4. Küçük deneme

Tam toplamadan önce üç tarihle sınanıyor: dosyalar iniyor mu, içerikleri dolu mu.

In [ ]:
!python egm_collector.py --url "https://egm.org.tr/bilgi-merkezi/istatistikler?r=b5eea773-1617-4344-8e66-1771226ee845" --code "GR-01-015" --out data/raw --limit 3

Aciliyor: https://egm.org.tr/bilgi-merkezi/istatistikler?r=b5eea773-1617-4344-8e66-1771226ee845
83 tarih bulundu.  Ilk: '29 Eylül 2026 Salı'  Son: '30 Aralık 2018 Pazar'
Excel dugmesi: BULUNDU
[1/3] 29 Eylül 2026 Salı ... -> GR-01-015_2026-09-29.xlsx
[2/3] 30 Ağustos 2026 Pazar ... -> GR-01-015_2026-08-30.xlsx
[3/3] 30 Temmuz 2026 Perşembe ... -> GR-01-015_2026-07-30.xlsx

Bitti.  indirilen: 3   atlanan: 0   hatali: 0
Klasor: /content/data/raw/GR-01-015


In [ ]:
import pandas as pd
from pathlib import Path

dosyalar = sorted(Path(f"data/raw/{"GR-01-015"}").glob("*.xlsx"))
print(f"{len(dosyalar)} dosya indi:")
for d in dosyalar:
    print(" ", d.name, f"({d.stat().st_size:,} byte)")

if dosyalar:
    ornek = pd.read_excel(dosyalar[0])
    print(f"\n{dosyalar[0].name}  ->  {ornek.shape[0]} satır × {ornek.shape[1]} kolon")
    display(ornek.head(10))

3 dosya indi:
  GR-01-015_2026-07-30.xlsx (12,470 byte)
  GR-01-015_2026-08-30.xlsx (12,509 byte)
  GR-01-015_2026-09-29.xlsx (12,542 byte)

GR-01-015_2026-07-30.xlsx  ->  274 satır × 3 kolon


,Unnamed: 0,Sonlanma Oranı (%),Yürürlüğe Giren Sözleşme Adedi
0,2003.10,55.41,388
1,2003.11,60.30,4715
2,2003.12,60.61,16983
3,2004.01,60.20,16284
4,2004.02,61.19,15917
5,2004.03,68.56,33662
6,2004.04,61.87,33077
7,2004.05,67.28,25660
8,2004.06,65.92,29553
9,2004.07,66.25,27635


## 5. Tam toplama

Tarih listesinin tamamı indiriliyor. Zaten inmiş dosyalar atlanıyor, yarıda
kesilirse tekrar çalıştırıldığında kaldığı yerden devam ediyor.

`--delay 2.5` istekler arasındaki bekleme; siteyi yormamak için bırakılmıştır.

In [ ]:
!python egm_collector.py --url "https://egm.org.tr/bilgi-merkezi/istatistikler?r=b5eea773-1617-4344-8e66-1771226ee845" --code "GR-01-015" --out data/raw --delay 2.5

Aciliyor: https://egm.org.tr/bilgi-merkezi/istatistikler?r=b5eea773-1617-4344-8e66-1771226ee845
83 tarih bulundu.  Ilk: '29 Eylül 2026 Salı'  Son: '30 Aralık 2018 Pazar'
Excel dugmesi: BULUNDU
[4/83] 29 Haziran 2026 Pazartesi ... -> GR-01-015_2026-06-29.xlsx
[5/83] 30 Mayıs 2026 Cumartesi ... -> GR-01-015_2026-05-30.xlsx
[6/83] 29 Nisan 2026 Çarşamba ... -> GR-01-015_2026-04-29.xlsx
[7/83] 30 Mart 2026 Pazartesi ... -> GR-01-015_2026-03-30.xlsx
[8/83] 27 Şubat 2026 Cuma ... -> GR-01-015_2026-02-27.xlsx
[9/83] 30 Ocak 2026 Cuma ... -> GR-01-015_2026-01-30.xlsx
[10/83] 30 Aralık 2025 Salı ... -> GR-01-015_2025-12-30.xlsx
[11/83] 29 Kasım 2025 Cumartesi ... -> GR-01-015_2025-11-29.xlsx
[12/83] 30 Ekim 2025 Perşembe ... -> GR-01-015_2025-10-30.xlsx
[13/83] 29 Eylül 2025 Pazartesi ... -> GR-01-015_2025-09-29.xlsx
[14/83] 30 Ağustos 2025 Cumartesi ... -> GR-01-015_2025-08-30.xlsx
[15/83] 30 Temmuz 2025 Çarşamba ... -> GR-01-015_2025-07-30.xlsx
[16/83] 29 Haziran 2025 Pazar ... -> GR-01-015_2

## 6. Diğer raporlar

Her raporun kendi adresi var. Aynı işlem, rapor kodu ve link değiştirilerek
tekrarlanıyor.

In [ ]:
# Örnek: şirket bazında fon büyüklükleri
RAPOR_URL_2 = ""   # <-- EGM-05-006'nın linki
RAPOR_KODU_2 = "EGM-05-006"

if RAPOR_URL_2:
    !python egm_collector.py --url "$RAPOR_URL_2" --code "$RAPOR_KODU_2" --out data/raw --delay 2.5

## 7. Toplananları indir

Colab oturumu kapandığında dosyalar siliniyor. İki seçenek: zip olarak indirmek
ya da Drive'a kopyalamak.

In [ ]:
# Seçenek A — zip'leyip bilgisayarına indir
from google.colab import files
!zip -rq egm_data.zip data/raw
files.download("egm_data.zip")

In [ ]:
# Seçenek B — Google Drive'a kaydet
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p "/content/drive/MyDrive/egm-bes-projesi"
# !cp -r data/raw "/content/drive/MyDrive/egm-bes-projesi/"

---

## Sonraki adım

Ham dosyalar toplandıktan sonra:

1. **Temizlik** — iki satıra bölünmüş başlıkların düzleştirilmesi, bilimsel
   gösterimdeki sayıların düzeltilmesi, uzun formata (`rapor_tarihi, şirket,
   segment, metrik, değer`) çevrilmesi.
2. **Ambar** — temiz tabloların PostgreSQL'e yüklenmesi, yıldız şema.
3. **Dashboard** — Tableau: kohort kalıcılık eğrisi, sonlanma nedenleri,
   pazar payı, maliyet karşılaştırması.
4. **Agent** — metrik sözlüğü, doğal dilden SQL, 30 soruluk doğruluk testi.